# Stage 5A — dense DINOv2 MIL and selective high-resolution revisit

Offline screening notebook. GPU T4 x2. First run smoke_studies=8, epochs=1;
then reset the session and use smoke_studies=0, epochs=20. Mount competition,
historical v5_labels.csv and YOUR historical v5s1 best_model_s42.pt (not 3D/4A).
Edit the next configuration cell for explicit file paths. No new pretraining weights.
Outputs mean/coarse/fine predictions; it does not create a ranked competition submission.
Gold was previously involved in label calibration and backbone selection: development only.
Attention locations are hypotheses, not verified lesion localizations.
Recovery: mount the PREVIOUS RUN OUTPUT containing stage5a_manifest.json,
stage5a_features/ and stage5a_*.pt; set resume_input below to that root.
Compatible caches and completed heads are reused. Stage results are exported early.
Budget exhaustion returns PAUSED normally with completed=False in the manifest.


In [ ]:
# ============================================================
# v4: Imports
# ============================================================
from __future__ import annotations
import gc, math, os, re, sys, time
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import pydicom
import cv2
from sklearn.metrics import roc_auc_score

IS_MAIN = True
print('Imports OK.')


In [ ]:
# ============================================================
# v5: Configuration — 288px/130mm 奈奎斯特分辨率 + v5 融合软标签 (teacher-student)
# ============================================================

TARGET_COLUMNS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA',
    'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture',
]
N_CLASSES = len(TARGET_COLUMNS)

# Soft-label columns
PROB_COLS   = [f'prob_{c}' for c in TARGET_COLUMNS]
WEIGHT_COLS = [f'weight_{c}' for c in TARGET_COLUMNS]
MASK_COLS   = [f'mask_{c}' for c in TARGET_COLUMNS]

# ---- 6 Clinical Slots ----
SLOTS = [
    ("SAG_FLUID_FS",   "Sagittal", True,  True),
    ("COR_FLUID_FS",   "Coronal",  True,  True),
    ("AX_FLUID_FS",    "Axial",    True,  True),
    ("SAG_FLUID_NOFS", "Sagittal", True,  False),
    ("COR_T1",         "Coronal",  False, False),
    ("SAG_T1",         "Sagittal", False, False),
]
N_SLOT = len(SLOTS)

# ---- Anatomical Priors ----
SLOT_PRIORS = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}

# ---- Diagnostic-specific TTA pooling ----
# 局部病灶用 max（保留最强信号），ACL/MCL 用 top2，弥漫性病变用 mean
# 与 0.91 notebook 的 TTA_TARGET_POOL 逐项一致
DIAG_POOL = {
    "Fracture": "max", "Contusion": "max",
    "Medial Meniscus": "max", "Lateral Meniscus": "max",
    "Baker's": "max",
    "ACL": "top2", "MCL": "top2",
    # ★ 0.91 同款: 仅用无 jitter 原始视图平均
    #   (jitter TTA 开启时生效; 关闭时所有视图皆原始, 等价于 mean)
    "Synovitis": "original_mean",
    # 其余（OA, Effusion）默认 mean
}

# ---- Jitter TTA 增广 (0.91 notebook augment() 移植) ----
AUG_ROT_DEG = 8.0          # 旋转 ±8°
AUG_SCALE = 0.08           # 缩放 +[0, 8%]
AUG_SHIFT = 0.05           # 平移 ±5%
AUG_INTENSITY = 0.1        # 强度 ±10%
AUG_SEED = 42              # 增广视图固定种子（确定性, 跨验证/测试/提交可复现）

CFG = dict(comp_input='/kaggle/input/competitions/rsna-knee-abnormality-detection',
           output_dir='/kaggle/working', dinov2_variant='vit_small_patch14_dinov2.lvd142m')
# EDIT PATHS HERE. Blank paths search /kaggle/input; ambiguous matches stop.
S5 = dict(v5_checkpoint='/kaggle/input/datasets/easoncyy/v5-bestmodel/best_model_s42.pt',
          labels='/kaggle/input/datasets/easoncyy/rsna-knee-v5-labels/v5_labels.csv',
          resume_input='',  # Directory containing the previous run's stage5a_manifest.json
          coarse_px=168, fine_px=280, slices=32,
          top_per_class=2, encode_batch=48, batch_size=32, epochs=20,
          head_lr=0.0003, seed=42, feature_minutes=300, smoke_studies=0,
          session_minutes=480, reserve_minutes=20)
N_GPUS=torch.cuda.device_count()
DEVICE=torch.device('cuda' if N_GPUS else 'cpu')
import random
random.seed(S5['seed']); np.random.seed(S5['seed']); torch.manual_seed(S5['seed'])
torch.set_num_threads(2)
print('Stage 5A', S5, 'GPUs', N_GPUS)


In [ ]:
# ============================================================
# v4: Slot Matching + Laterality Detection + DICOM Header Annotation
# ============================================================

# ---- DICOM Header Annotation (Ref1: annotate_sequences) ----
_SEP = re.compile(r'[_\-.]')
_FATSAT_RX = re.compile(
    r'\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|'
    r'water excit|\btirm\b|\bsting\b|\bfatsup\b'
)
_T1_RX = re.compile(r'\bt1\b|\bt1w\b')
_T2_RX = re.compile(r'\bt2\b|\bt2w\b')
_PD_RX = re.compile(r'\bpd\b|\bpdw\b|proton|\bdp\b|dens')

FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}

_HDR_TAGS = [
    'SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence',
    'RepetitionTime', 'EchoTime', 'Laterality', 'ImageLaterality',
    'ImagePositionPatient', 'PixelSpacing',
]


def _tag_side(group):
    """从 DICOM Laterality 标签推断侧性。"""
    values = [str(x).strip().upper() for x in group.get('Laterality', pd.Series(dtype=object)).dropna()]
    if 'ImageLaterality' in group.columns:
        values += [str(x).strip().upper() for x in group['ImageLaterality'].dropna()]
    values = [x[0] for x in values if x and x[0] in ('L', 'R')]
    return values[0] if values else None


def _position_side(group, min_offset_mm=5.0):
    """从 ImagePositionPatient[0] 推断侧性：DICOM LPS 中 +x = 患者左侧。"""
    xs = []
    for raw in group.get('ImagePositionPatient', pd.Series(dtype=object)).dropna():
        try:
            xs.append(float(str(raw).split('|')[0]))
        except Exception:
            pass
    if not xs:
        return None
    median_x = float(np.median(xs))
    if abs(median_x) < min_offset_mm:
        return None
    return 'R' if median_x < 0 else 'L'


def detect_laterality(headers_df):
    """为每个 study 确定侧性（左/右），结合标签和几何位置。"""
    tagged, positioned = {}, {}
    for study_uid, group in headers_df.groupby('StudyInstanceUID'):
        tagged[study_uid] = _tag_side(group)
        positioned[study_uid] = _position_side(group)

    comparable = [s for s in tagged if tagged[s] and positioned[s]]
    agreement = float(np.mean([
        tagged[s] == positioned[s] for s in comparable
    ])) if comparable else np.nan

    use_position = bool(comparable) and np.isfinite(agreement) and agreement >= 0.85

    resolved = {
        uid: (tagged[uid] or (positioned[uid] if use_position else None))
        for uid in tagged
    }
    coverage = float(np.mean([v is not None for v in resolved.values()]))

    if IS_MAIN:
        print(f'Laterality: tag_coverage={len([v for v in tagged.values() if v])/max(len(tagged),1):.1%}, '
              f'agreement={agreement:.1%} on {len(comparable)} studies, '
              f'final_coverage={coverage:.1%}')
    return resolved


def annotate_sequences(df):
    """从 DICOM header 推断 Fluid/FatSat/Weight，作为 train_series.csv 的 fallback。"""
    df = df.copy()

    # Fat suppression detection
    desc = (df.get('SeriesDescription', '').fillna('') + ' ' +
            df.get('SequenceName', '').fillna(''))
    desc = desc.str.lower().str.replace(_SEP, ' ', regex=True)

    scan_options = df.get('ScanOptions', '').fillna('').str.upper().str.split('|')
    option_fatsat = scan_options.apply(
        lambda tokens: any(t.strip() in FATSAT_OPTS for t in tokens))
    df['fatsat_detected'] = desc.str.contains(_FATSAT_RX) | option_fatsat

    # Weight detection
    tr = pd.to_numeric(df.get('RepetitionTime', np.nan), errors='coerce')
    te = pd.to_numeric(df.get('EchoTime', np.nan), errors='coerce')
    named_t1 = desc.str.contains(_T1_RX)
    named_t2 = desc.str.contains(_T2_RX)
    named_pd = desc.str.contains(_PD_RX)

    df['weight'] = np.where(
        named_t1 & ~named_t2 & ~named_pd, 'T1',
        np.where(named_t2 & ~named_pd, 'T2',
                 np.where(named_pd, 'PD',
                          np.where(tr < 800, 'T1',
                                   np.where(te > 60, 'T2',
                                            np.where(tr >= 800, 'PD', 'UNK'))))))
    df['fluid_detected'] = df['weight'].isin(['PD', 'T2'])

    return df


# ---- Slot Matching ----
def match_slots_for_study(study_series_df):
    """为单个 study 的每个 slot 匹配最优 series。"""
    slots_found = {}
    for slot_name, plane, fluid, fatsat in SLOTS:
        candidates = study_series_df[
            (study_series_df['Anatomical_Plane'] == plane)
            & (study_series_df['Fluid_Sensitive'] == (1 if fluid else 0))
            & (study_series_df['Fat_Suppression'] == (1 if fatsat else 0))
        ]
        if len(candidates) == 0 and not fluid:
            candidates = study_series_df[
                (study_series_df['Anatomical_Plane'] == plane)
                & (study_series_df['Fluid_Sensitive'] == 0)
            ]
        if len(candidates) > 0:
            best = candidates.sort_values('n_slices', ascending=False).iloc[0]
            slots_found[slot_name] = {
                'series_uid': best['SeriesInstanceUID'],
                'dir': best['dir'],
                'n_slices': int(best['n_slices']),
                'plane': plane,
            }
        else:
            slots_found[slot_name] = None
    return slots_found


def build_study_slot_map(series_meta, dicom_root):
    """为所有 study 构建 slot→series 映射。"""
    df = series_meta.copy()
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    df['SeriesInstanceUID'] = df['SeriesInstanceUID'].astype(str)

    # 计算 DICOM 目录和切片数
    dirs, n_slices_list = [], []
    for _, row in df.iterrows():
        d = str(dicom_root / row['StudyInstanceUID'] / row['SeriesInstanceUID'])
        dirs.append(d)
        if os.path.isdir(d):
            files = [f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))]
            n_dcm = len([f for f in files if f.endswith('.dcm')])
            if n_dcm == 0:
                n_dcm = len([f for f in files if not f.startswith('.')])
            n_slices_list.append(n_dcm)
        else:
            n_slices_list.append(0)
    df['dir'] = dirs
    df['n_slices'] = n_slices_list

    slot_map, study_series_map = {}, {}
    for study_uid, grp in df.groupby('StudyInstanceUID'):
        study_series_map[study_uid] = grp
        slot_map[study_uid] = match_slots_for_study(grp)

    # 统计
    slot_counts = {}
    for slots in slot_map.values():
        for name, sid in slots.items():
            slot_counts[name] = slot_counts.get(name, 0) + (1 if sid is not None else 0)

    if IS_MAIN:
        n_studies = len(slot_map)
        print(f'Slot map: {n_studies} studies')
        for name, count in slot_counts.items():
            print(f'  {name:<18s}: {count:5d}/{n_studies} ({count/n_studies*100:.0f}%)')

    return slot_map, study_series_map

print('Slot matching v4 ready.')


In [ ]:
# ============================================================
# v4: DICOM I/O — 空间排序 + 物理裁剪 + 侧性归一化 + 并行读取
# ============================================================

# ---- 空间切片排序 (Ref2: dominant_axis) ----
PLANE_AXIS = {"Sagittal": 0, "Coronal": 1, "Axial": 2}

def _list_dcm_files(series_dir):
    """列出 DICOM 文件（不依赖 .dcm 扩展名，竞赛 test 集无后缀）。"""
    sd = Path(series_dir)
    if not sd.is_dir():
        return []
    all_files = sorted(f.name for f in sd.iterdir() if f.is_file())
    dcm = [f for f in all_files if f.endswith('.dcm')]
    return dcm if dcm else [f for f in all_files if not f.startswith('.')]

def spatially_sorted_files(series_dir, plane=None):
    """按 ImagePositionPatient 在切片法线方向上的投影排序。
    文件名排序的 Spearman 相关系数仅 0.009——完全随机。
    """
    series_dir = Path(series_dir)
    files = _list_dcm_files(series_dir)
    if not files:
        return []

    axis = PLANE_AXIS.get(plane, 2)
    rows = []
    for fname in files:
        try:
            ds = pydicom.dcmread(
                str(series_dir / fname), stop_before_pixels=True, force=True,
                specific_tags=['ImagePositionPatient', 'InstanceNumber'])
            ipp = getattr(ds, 'ImagePositionPatient', None)
            instance = getattr(ds, 'InstanceNumber', None)
            if ipp is not None and len(ipp) >= 3:
                candidate = np.array(ipp[:3], dtype=np.float64)
                pos = float(candidate[axis]) if np.isfinite(candidate).all() else None
            else:
                pos = None
            inst_val = float(instance) if instance is not None else None
        except Exception:
            pos, inst_val = None, None
        rows.append((fname, pos, inst_val))

    positioned = [r for r in rows if r[1] is not None]
    threshold = max(2, int(0.8 * len(rows)))

    if len(positioned) >= threshold:
        # 主排序：通过平面坐标
        rows.sort(key=lambda r: (
            r[1] if r[1] is not None else 0.0,
            r[2] if r[2] is not None else float('inf'),
        ))
    elif sum(r[2] is not None for r in rows) >= threshold:
        rows.sort(key=lambda r: (
            r[2] if r[2] is not None else float('inf'),
        ))
    # else: 保持文件名顺序

    return [r[0] for r in rows]


# ---- 侧性归一化 ----
def normalise_laterality(image, plane, laterality):
    """右膝映射为左膝：冠/轴面水平翻转，矢面反转切片顺序。"""
    if laterality != 'R':
        return image
    # image: [N_slices, H, W] numpy
    if plane in ('Coronal', 'Axial'):
        return np.flip(image, axis=-1).copy()  # 水平翻转
    else:
        return np.flip(image, axis=0).copy()    # 反转切片顺序


# ---- 物理裁剪 ----
def physical_crop(volume, px, crop_mm=160.0):
    """基于 PixelSpacing 裁剪到固定物理 FOV，消除不同扫描仪的空间尺度差异。"""
    if px is None or not np.isfinite(px) or px <= 0:
        return volume
    desired = int(round(crop_mm / px))
    h, w = volume.shape[1], volume.shape[2]
    if not (16 < desired < min(h, w)):
        return volume
    cy, cx = h // 2, w // 2
    half = desired // 2
    return volume[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]


# ---- 读取单 series 为 volume ----
def read_series_volume(series_dir, plane=None, laterality=None,
                       image_size=224, crop_mm=160.0):
    """读取 DICOM 序列 → 空间排序 → 物理裁剪 → 侧性归一化 → 归一化 → 缩放。"""
    sorted_files = spatially_sorted_files(series_dir, plane)
    if not sorted_files:
        return None, None

    series_dir = Path(series_dir)
    slices_info = []
    px = None

    for fname in sorted_files:
        try:
            ds = pydicom.dcmread(str(series_dir / fname), force=True)
            img = ds.pixel_array.astype(np.float32)

            # Rescale
            slope = float(getattr(ds, 'RescaleSlope', 1) or 1)
            intercept = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            img = img * slope + intercept

            # PixelSpacing (取第一个有效值)
            if px is None:
                ps = getattr(ds, 'PixelSpacing', None)
                if ps is not None and len(ps) >= 1:
                    try:
                        px = float(ps[0])
                    except Exception:
                        pass

            slices_info.append(img)
        except Exception:
            raise RuntimeError(f'Pixel decode failed: {series_dir / fname}')

    if not slices_info:
        return None, None

    volume = np.stack(slices_info, axis=0)  # [N, H, W]

    # 物理裁剪
    volume = physical_crop(volume, px, crop_mm)

    # 侧性归一化
    volume = normalise_laterality(volume, plane, laterality)

    # 鲁棒归一化 (1st-99th percentile)
    v_low, v_high = np.percentile(volume, [1.0, 99.0])
    volume = np.clip(volume, v_low, v_high)
    denom = max(v_high - v_low, 1e-6)
    volume = (volume - v_low) / denom

    # 缩放到 target size
    resized = []
    for img in volume:
        r = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
        resized.append(r)
    return np.stack(resized, axis=0).astype(np.float32), px


# ---- 缓存切片采样 ----
def sample_cache_slices(volume, n_cache=9, center_pct=(0.2, 0.8)):
    """从 volume 的 central 60% 区域均匀采样 n_cache 个切片。"""
    n_total = volume.shape[0]
    if n_total <= n_cache:
        indices = list(range(n_total))
        while len(indices) < n_cache:
            indices.append(indices[-1])
        return volume[np.array(indices)]

    low = int(center_pct[0] * (n_total - 1))
    high = int(center_pct[1] * (n_total - 1))
    if high <= low:
        low, high = 0, n_total - 1
    indices = np.unique(np.linspace(low, high, n_cache).astype(int))
    while len(indices) < n_cache:
        indices = np.append(indices, indices[-1])
    return volume[indices[:n_cache]]

print('DICOM I/O v4 ready.')


In [ ]:
"""Stage 5A reusable tensor operations; no data loading or network side effects."""
import numpy as np
import torch
from torch import nn


def window_centers(n, limit=32):
    if n < 3:
        return np.empty(0, dtype=np.int64)
    return np.unique(np.linspace(1, n - 2, min(limit, n - 2)).round().astype(np.int64))


class DenseMIL(nn.Module):
    def __init__(self, dim=1152, hidden=128, classes=12, slots=6, pooling='attention'):
        super().__init__()
        self.pooling = pooling
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot = nn.Embedding(slots, hidden)
        self.scale = nn.Embedding(2, hidden)
        self.position = nn.Linear(1, hidden, bias=False)
        self.a = nn.Linear(hidden, hidden)
        self.b = nn.Linear(hidden, hidden)
        self.query = nn.Linear(hidden, classes, bias=False)
        self.out = nn.Parameter(torch.randn(classes, hidden) * .02)
        self.bias = nn.Parameter(torch.zeros(classes))
        self.drop = nn.Dropout(.2)

    def forward(self, x, mask, slot, pos, scale, return_attention=False):
        h = self.proj(x) + self.slot(slot) + self.position(pos.unsqueeze(-1)) + self.scale(scale)
        score = self.query(torch.tanh(self.a(h)) * torch.sigmoid(self.b(h))).transpose(1, 2)
        if self.pooling == 'mean':
            score = torch.zeros_like(score)
        valid = mask[:, None, :].bool()
        attention = score.masked_fill(~valid, -1e4).softmax(-1) * valid
        attention = attention / attention.sum(-1, keepdim=True).clamp_min(1e-8)
        context = torch.einsum('bct,bth->bch', attention, h)
        logits = (self.drop(context) * self.out).sum(-1) + self.bias
        return (logits, attention) if return_attention else logits


def selected_tokens(attention, mask, per_class=2):
    """At most 24 coarse tokens, chosen from image attention only, never labels."""
    valid = np.flatnonzero(mask)
    if not len(valid):
        return np.empty(0, dtype=np.int64)
    chosen = []
    for row in attention:
        chosen.extend(valid[np.argsort(-row[valid], kind='stable')[:per_class]])
    return np.unique(chosen)


def weighted_bce(logits, targets, weights, masks):
    weight = weights * masks
    return (nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none') * weight).sum() / weight.sum().clamp_min(1e-8)


In [ ]:
# Stage 5A: streamed DICOM -> frozen features -> mean/MIL/coarse-to-fine controls.
import hashlib, json, copy, shutil, zipfile

session_started = time.monotonic()

def atomic_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2), encoding='utf-8')
    tmp.replace(path)

def compatible_run(previous, current):
    if previous.get('experiment') != 'stage5a':
        raise ValueError('Resume manifest is not Stage 5A')
    for key in ('checkpoint_sha256', 'labels_sha256'):
        if not previous.get(key) or previous[key] != current[key]:
            raise ValueError(f'Resume mismatch: {key}')
    # Paths, wall-clock budgets and smoke size do not change image features.
    for key in ('coarse_px','fine_px','slices','top_per_class','seed','epochs','head_lr','batch_size'):
        if previous['config'].get(key) != current['config'].get(key):
            raise ValueError(f'Resume configuration mismatch: {key}')
    if previous.get('feature_contract', 'stage5a-v1') != 'stage5a-v1':
        raise ValueError('Unknown feature preprocessing contract')

def validated_cache(path, phase, selected=None):
    try:
        with np.load(path, allow_pickle=False) as z:
            keys=('x','mask','slot','pos','center','scale')
            if not set(keys).issubset(z.files): return False
            b={k:z[k] for k in keys}
        n=len(b['mask'])
        expected=N_SLOT*S5['slices'] if phase=='coarse' else max(1,len(selected))
        if n!=expected or b['x'].shape!=(n,1152) or b['x'].dtype!=np.float16: return False
        if any(b[k].shape!=(n,) for k in keys[1:]): return False
        if b['mask'].dtype!=np.bool_ or not b['mask'].any(): return False
        if any(b[k].dtype!=np.int64 for k in ('slot','center','scale')): return False
        if not all(np.isfinite(b[k]).all() for k in keys): return False
        if not ((b['slot']>=0)&(b['slot']<N_SLOT)).all(): return False
        if not ((b['pos']>=0)&(b['pos']<=1)).all(): return False
        if not (b['scale']==int(phase=='fine')).all(): return False
        valid=b['mask']
        if (b['center'][valid]<1).any(): return False
        if phase=='fine':
            wanted=np.asarray(selected,dtype=np.int64)
            actual=np.stack([b['slot'],b['center']],axis=1)
            if not np.array_equal(actual[valid],wanted[valid]): return False
        return True
    except (OSError,ValueError,KeyError,EOFError,TypeError,IndexError,zipfile.BadZipFile):
        return False

def restore_cache(uid, phase, selected=None):
    dest=cache_path(uid,phase)
    for root in resume_roots:
        src=root/'stage5a_features'/phase/dest.name
        if not src.is_file() or not validated_cache(src,phase,selected): continue
        if src.resolve()!=dest.resolve():
            dest.parent.mkdir(parents=True,exist_ok=True)
            tmp=dest.with_suffix('.resume.tmp')
            shutil.copyfile(src,tmp); tmp.replace(dest)
        return True
    return False

def session_budget_reached():
    # Reserve time for reporting; never shorten a test set to meet the budget.
    return time.monotonic()-session_started >= (S5.get('session_minutes',480)-S5.get('reserve_minutes',20))*60

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def find_asset(explicit, filename):
    if explicit and Path(explicit).is_file():
        return Path(explicit)
    hits = list(Path('/kaggle/input').rglob(filename))
    if len(hits) != 1:
        raise FileNotFoundError(f'Set the explicit path for {filename}; found {hits}')
    return hits[0]

output_dir = Path(CFG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), 'Stage 5A feature extraction requires a GPU.'
asset = find_asset(S5['v5_checkpoint'], 'best_model_s42.pt')
label_asset = find_asset(S5['labels'], 'v5_labels.csv')
assert sha256(label_asset) == 'c13adffaabf4f8e518abb038282bb1aa09baac7652a9165e030710c457d0be6a', 'Use historical v5 labels'
# Only load your own trusted v5 checkpoint. It contains configuration as well as tensors.
checkpoint = torch.load(asset, map_location='cpu', weights_only=False)
assert checkpoint['targets'] == TARGET_COLUMNS
assert checkpoint['config']['seed'] == 42 and checkpoint['config']['image_size'] == 288
state = {k.removeprefix('module.'): v for k, v in checkpoint['model'].items()}
for k, v in (checkpoint.get('ema') or {}).get('shadow', {}).items():
    state[k.removeprefix('module.')] = v
backbone_state = {k[len('dinov2.'):]: v for k, v in state.items() if k.startswith('dinov2.')}
assert backbone_state, 'No v5 DINOv2 backbone found'
encoders = {}
for size in (S5['coarse_px'], S5['fine_px']):
    enc = timm.create_model(CFG['dinov2_variant'], pretrained=False, num_classes=0, img_size=size)
    sd = dict(backbone_state)
    p = sd['pos_embed']; old = math.isqrt(p.shape[1] - 1); new = math.isqrt(enc.pos_embed.shape[1] - 1)
    assert old * old == p.shape[1] - 1
    grid = p[:, 1:].reshape(1, old, old, -1).permute(0, 3, 1, 2)
    grid = F.interpolate(grid, (new, new), mode='bicubic', align_corners=False, antialias=True)
    sd['pos_embed'] = torch.cat([p[:, :1], grid.permute(0, 2, 3, 1).reshape(1, new * new, -1)], 1)
    enc.load_state_dict(sd, strict=True)
    enc.requires_grad_(False).eval().to(DEVICE)
    if N_GPUS > 1:
        class Tokens(nn.Module):
            def __init__(self, model):
                super().__init__(); self.model = model
            def forward(self, x):
                return self.model.forward_features(x)
        enc = nn.DataParallel(Tokens(enc))
    encoders[size] = enc
audit = dict(experiment='stage5a', config=S5, checkpoint_sha256=sha256(asset), labels_sha256=sha256(label_asset),
             gold_is_development_only=True, parent_version_id=347718768, parent_comparison='pending')
resume_roots=[]
for root in [output_dir] + ([Path(S5['resume_input'])] if S5.get('resume_input') else []):
    manifest=root/'stage5a_manifest.json'
    if manifest.is_file():
        compatible_run(json.loads(manifest.read_text(encoding='utf-8')),audit)
        if root.resolve() not in [r.resolve() for r in resume_roots]: resume_roots.append(root)
    elif root!=output_dir:
        raise FileNotFoundError(f'Resume root must contain {manifest.name}: {root}')
    elif (root/'stage5a_features').exists():
        raise RuntimeError('Existing feature cache has no provenance manifest; use a clean output directory')
audit.update(feature_contract='stage5a-v1',completed=False,status='running',resume_sources=[str(r) for r in resume_roots])
atomic_json(output_dir / 'stage5a_manifest.json',audit)
print('Resume roots:',resume_roots,flush=True)
del checkpoint, state, backbone_state
gc.collect()

@torch.inference_mode()
def encode(windows, size):
    outputs = []
    enc = encoders[size]
    for start in range(0, len(windows), S5['encode_batch']):
        x = torch.from_numpy(np.stack(windows[start:start+S5['encode_batch']])).to(DEVICE).float() / 255
        mean = x.new_tensor([.485,.456,.406])[None,:,None,None]
        std = x.new_tensor([.229,.224,.225])[None,:,None,None]
        with torch.autocast('cuda', dtype=torch.float16):
            f = enc((x-mean)/std) if isinstance(enc, nn.DataParallel) else enc.forward_features((x-mean)/std)
            patches = f[:,1:]
            feat = torch.cat([f[:,0], patches.mean(1), patches.topk(max(1, patches.shape[1]//8),dim=1).values.mean(1)],1)
        outputs.append(feat.float().cpu().numpy().astype(np.float16))
    return np.concatenate(outputs)

def cache_path(uid, phase):
    return output_dir / 'stage5a_features' / phase / (hashlib.sha256(str(uid).encode()).hexdigest()+'.npz')

def read_slot(info, plane):
    # DICOM decode errors are surfaced rather than converted to fake zero slices.
    lat = None
    files = _list_dcm_files(info['dir'])
    if files:
        ds = pydicom.dcmread(str(Path(info['dir'])/files[0]), stop_before_pixels=True, force=True)
        lat = str(getattr(ds, 'ImageLaterality', '') or getattr(ds, 'Laterality', '')).upper()
    volume, _ = read_series_volume(info['dir'], plane=plane, laterality=lat,
                                    image_size=S5['fine_px'], crop_mm=130.)
    return volume

def extract_study(uid, mapping, phase, selected=None):
    # Fine selections reference actual, physically ordered slice indices.
    n = N_SLOT*S5['slices'] if selected is None else len(selected)
    n = max(1,n)
    x=np.zeros((n,1152),np.float16); mask=np.zeros(n,bool)
    slots=np.zeros(n,np.int64); pos=np.zeros(n,np.float32); centers=np.zeros(n,np.int64)
    size=S5['coarse_px'] if selected is None else S5['fine_px']
    windows=[]; dest=[]; failures=[]
    for s,(name,plane,_,_) in enumerate(SLOTS):
        info=mapping.get(name)
        if info is None: continue
        if selected is not None and not any(int(z[0])==s for z in selected): continue
        try:
            vol=read_slot(info,plane)
            if vol is None or len(vol)<3: raise ValueError('No usable volume')
            candidates=window_centers(len(vol),S5['slices'])
            rows=[(s*S5['slices']+j,int(c)) for j,c in enumerate(candidates)] if selected is None else [(j,int(z[1])) for j,z in enumerate(selected) if int(z[0])==s]
            for j,c in rows:
                if not 1<=c<len(vol)-1: raise ValueError('Fine center outside volume')
                w=vol[c-1:c+2]
                if size!=S5['fine_px']:
                    w=np.stack([cv2.resize(a,(size,size),interpolation=cv2.INTER_AREA) for a in w])
                windows.append((w*255).clip(0,255).round().astype(np.uint8)); dest.append(j)
                slots[j]=s; pos[j]=c/max(1,len(vol)-1); centers[j]=c
        except Exception as e:
            failures.append(dict(uid=uid,slot=name,error=str(e)))
    if windows:
        x[dest]=encode(windows,size); mask[dest]=True
    if not mask.any(): raise RuntimeError(f'No valid images for {uid}: {failures}')
    p=cache_path(uid,phase); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_suffix('.tmp.npz')
    np.savez_compressed(tmp,x=x,mask=mask,slot=slots,pos=pos,center=centers,scale=np.full(n,int(selected is not None),np.int64))
    tmp.replace(p)
    BANKS.pop((uid,phase),None)
    return failures

BANKS = {}

def load_bank(uid, fine=False):
    def get(phase):
        key=(uid,phase)
        if key not in BANKS:
            with np.load(cache_path(uid,phase)) as z: BANKS[key]={k:z[k] for k in z.files}
        return BANKS[key]
    bank=get('coarse')
    if fine:
        extra=get('fine')
        bank={k:np.concatenate([v,extra[k]]) for k,v in bank.items()}
    return bank

def tensor_batch(uids,fine=False):
    banks=[load_bank(u,fine) for u in uids]; n=max(len(b['mask']) for b in banks)
    out=[]
    for key in ('x','mask','slot','pos','scale'):
        arr=np.stack([np.pad(b[key],[(0,n-len(b[key]))]+([(0,0)] if key=='x' else [])) for b in banks])
        t=torch.from_numpy(arr).to(DEVICE)
        out.append(t.float() if key in ('x','pos') else t)
    return out

def fit_head(uids, labels, name, initial=None, pooling='attention', fine=False):
    torch.manual_seed(S5['seed']); rng=np.random.default_rng(S5['seed'])
    head=DenseMIL(pooling=pooling).to(DEVICE)
    if initial is not None: head.load_state_dict(initial.state_dict(),strict=True)
    optimizer=torch.optim.AdamW(head.parameters(),lr=S5['head_lr'],weight_decay=1e-3)
    history=[]
    # Fixed epochs: gold labels never choose epochs or fine selection locations.
    for epoch in range(S5['epochs']):
        head.train(); order=rng.permutation(uids); total=0.; steps=0
        for start in range(0,len(order),S5['batch_size']):
            ids=list(order[start:start+S5['batch_size']]); batch=tensor_batch(ids,fine)
            row=labels.loc[ids]
            y,w,m=[torch.tensor(row[c].to_numpy(np.float32),device=DEVICE) for c in (PROB_COLS,WEIGHT_COLS,MASK_COLS)]
            optimizer.zero_grad(set_to_none=True)
            loss=weighted_bce(head(*batch),y,w,m); loss.backward()
            nn.utils.clip_grad_norm_(head.parameters(),1.)
            optimizer.step(); total+=float(loss.detach()); steps+=1
        history.append(dict(epoch=epoch+1,loss=total/max(1,steps)))
        print(name,history[-1],flush=True)
    head.eval()
    torch.save(dict(state_dict=head.cpu().state_dict(),config=S5,pooling=pooling,fine=fine,audit=audit),output_dir/(name+'.pt'))
    head.to(DEVICE)
    pd.DataFrame(history).to_csv(output_dir/(name+'_history.csv'),index=False)
    return head

def get_head(uids, labels, name, initial=None, pooling='attention', fine=False):
    uid_hash=hashlib.sha256('\n'.join(sorted(uids)).encode()).hexdigest()
    for root in resume_roots:
        path=root/(name+'.pt')
        if not path.is_file(): continue
        # The mounted run is explicitly selected by the user; load its trusted head.
        ckpt=torch.load(path,map_location='cpu',weights_only=False)
        compatible_run(ckpt['audit'],audit)
        if ckpt.get('pooling')!=pooling or ckpt.get('fine')!=fine: continue
        old_config=ckpt['config']
        if old_config.get('smoke_studies',0)!=S5.get('smoke_studies',0): continue
        # Legacy full runs use all non-Gold train UIDs, same competition and protocol.
        if ckpt.get('train_uid_sha256',uid_hash)!=uid_hash: continue
        head=DenseMIL(pooling=pooling).to(DEVICE)
        head.load_state_dict(ckpt['state_dict'],strict=True); head.eval()
        dest=output_dir/path.name
        if path.resolve()!=dest.resolve(): shutil.copyfile(path,dest)
        history=root/(name+'_history.csv')
        if history.is_file() and history.resolve()!=(output_dir/history.name).resolve(): shutil.copyfile(history,output_dir/history.name)
        print('Restored completed head:',name,flush=True)
        return head
    head=fit_head(uids,labels,name,initial,pooling,fine)
    path=output_dir/(name+'.pt')
    ckpt=torch.load(path,map_location='cpu',weights_only=False)
    ckpt['train_uid_sha256']=uid_hash
    torch.save(ckpt,path)
    return head

@torch.inference_mode()
def predict_head(head,uids,fine=False):
    head.eval(); outputs=[]
    for start in range(0,len(uids),S5['batch_size']):
        outputs.append(head(*tensor_batch(uids[start:start+S5['batch_size']],fine)).sigmoid().cpu().numpy())
    return np.concatenate(outputs)

comp_input=Path(CFG['comp_input'])
if not (comp_input/'train.csv').is_file():
    candidates=[Path('/kaggle/input/rsna-knee-abnormality-detection'),Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')]
    comp_input=next(p for p in candidates if (p/'train.csv').is_file())
train_meta=pd.read_csv(comp_input/'train.csv',dtype={'StudyInstanceUID':str})
test_meta=pd.read_csv(comp_input/'test.csv',dtype={'StudyInstanceUID':str})
gold=train_meta.loc[train_meta[TARGET_COLUMNS].notna().all(1)].set_index('StudyInstanceUID')
train_uids=sorted(set(train_meta.StudyInstanceUID)-set(gold.index)); gold_uids=sorted(gold.index); test_uids=test_meta.StudyInstanceUID.tolist()
labels=pd.read_csv(label_asset,dtype={'StudyInstanceUID':str}).set_index('StudyInstanceUID')
assert labels.index.is_unique and set(train_uids)<=set(labels.index)
values=labels.loc[train_uids,PROB_COLS+WEIGHT_COLS+MASK_COLS].to_numpy(float)
assert np.isfinite(values).all() and (values>=0).all() and (values<=1).all()
assert not set(train_uids)&set(gold_uids) and not set(train_uids+gold_uids)&set(test_uids)
if S5['smoke_studies']:
    train_uids=train_uids[:S5['smoke_studies']]; gold_uids=gold_uids[:4]
maps={}
for split,ids in [('train',train_uids+gold_uids),('test',test_uids)]:
    metadata=pd.read_csv(comp_input/(split+'_series.csv'),dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
    part,_=build_study_slot_map(metadata.loc[metadata.StudyInstanceUID.isin(ids)],comp_input/(split+'_series'))
    maps.update(part)
all_uids=train_uids+gold_uids+test_uids

def export_head(name,head,metrics):
    pred=predict_head(head,gold_uids,fine=name=='fine')
    df=pd.DataFrame(pred,columns=TARGET_COLUMNS); df.insert(0,'StudyInstanceUID',gold_uids)
    df.to_csv(output_dir/f'gold_stage5a_{name}.csv',index=False)
    aucs=[]
    for j,c in enumerate(TARGET_COLUMNS):
        y=gold.loc[gold_uids,c].to_numpy()
        auc=roc_auc_score(y,pred[:,j]) if len(np.unique(y))==2 else float('nan')
        metrics.append(dict(model=name,target=c,auc=auc)); aucs.append(auc)
    print(name,'Gold development macro',np.nanmean(aucs))
    test_pred=predict_head(head,test_uids,fine=name=='fine')
    df=pd.DataFrame(test_pred,columns=TARGET_COLUMNS); df.insert(0,'StudyInstanceUID',test_uids)
    assert np.isfinite(test_pred).all() and df.StudyInstanceUID.tolist()==test_uids
    df.to_csv(output_dir/f'submission_stage5a_{name}.csv',index=False)
    pd.DataFrame(metrics).to_csv(output_dir/'stage5a_auc.csv',index=False)
    gold.loc[gold_uids,TARGET_COLUMNS].to_csv(output_dir/'stage5a_gold_truth.csv')

def run_stage5a():
    failures=[]; selection_log=[]; metrics=[]; heads={}
    reused=dict(coarse=0,fine=0); computed=dict(coarse=0,fine=0)
    for root in resume_roots:
        p=root/'stage5a_decode_failures.json'
        if p.is_file(): failures.extend(json.loads(p.read_text(encoding='utf-8')))

    def persist(stage,done=0,paused=False,completed=False):
        audit.update(status='paused' if paused else ('completed' if completed else 'running'),
                     completed=completed,stage=stage,processed=done,total=len(all_uids),
                     reused=reused,computed=computed,exported_heads=list(heads),
                     train_studies=len(train_uids),gold_studies=len(gold_uids),test_studies=len(test_uids),
                     runtime_minutes=(time.monotonic()-session_started)/60,smoke=bool(S5['smoke_studies']))
        atomic_json(output_dir/'stage5a_manifest.json',audit)
        atomic_json(output_dir/'stage5a_decode_failures.json',failures)
        atomic_json(output_dir/'stage5a_selected_slices.json',selection_log)
        if paused:
            print(f'PAUSED at {stage} {done}/{len(all_uids)}. Existing predictions and caches are saved. '
                  'Mount this output as resume_input in a new run; do not submit a partial run.',flush=True)

    persist('coarse')
    coarse_started=time.monotonic()
    for i,uid in enumerate(all_uids):
        if session_budget_reached() or time.monotonic()-coarse_started>S5['feature_minutes']*60:
            persist('coarse',i,paused=True); return
        if restore_cache(uid,'coarse'): reused['coarse']+=1
        else:
            failures.extend(extract_study(uid,maps.get(uid,{}),'coarse')); computed['coarse']+=1
        if (i+1)%100==0:
            print(f'coarse {i+1}/{len(all_uids)}; reused {reused["coarse"]}',flush=True)
            persist('coarse',i+1)
    # Evaluate every completed control BEFORE starting the costly fine pass.
    for name,pooling in [('mean','mean'),('coarse','attention'),('coarse_continue','attention')]:
        if session_budget_reached(): persist('heads',len(heads),paused=True); return
        head=get_head(train_uids,labels,'stage5a_'+name,
                      initial=heads.get('coarse') if name=='coarse_continue' else None,pooling=pooling)
        export_head(name,head,metrics); heads[name]=head; persist('heads',len(heads))
    persist('fine')
    with torch.inference_mode():
        for i,uid in enumerate(all_uids):
            if session_budget_reached(): persist('fine',i,paused=True); return
            bank=load_bank(uid)
            _,att=heads['coarse'](*tensor_batch([uid]),return_attention=True)
            selected=selected_tokens(att[0].cpu().numpy(),bank['mask'],S5['top_per_class'])
            coordinates=[(int(bank['slot'][j]),int(bank['center'][j])) for j in selected]
            if restore_cache(uid,'fine',coordinates): reused['fine']+=1
            else:
                failures.extend(extract_study(uid,maps.get(uid,{}),'fine',coordinates)); computed['fine']+=1
            selection_log.append(dict(uid=uid,slot_center=coordinates))
            if (i+1)%100==0:
                print(f'fine {i+1}/{len(all_uids)}; reused {reused["fine"]}',flush=True)
                persist('fine',i+1)
    if session_budget_reached(): persist('fine_head',len(all_uids),paused=True); return
    head=get_head(train_uids,labels,'stage5a_fine',initial=heads['coarse'],fine=True)
    export_head('fine',head,metrics); heads['fine']=head
    persist('complete',len(all_uids),completed=True)
    print('Screening complete. Compare with parent936 before creating submission.csv. No automatic leaderboard submission.')

run_stage5a()
